# m55 lab · Capstone: a multimodal image-guided procedure on a digital phantom

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

One synthetic left atrium goes through the whole chain: a CT-like volume, segmentation, a surface, registration to
simulated map points, prediction of the error at targets, an overlay on a simulated fluoroscopy view, the same steps
on real anatomy, and an error and dose budget at the end.

| Part | What you do | Data |
| --- | --- | --- |
| A | C-arm projection and magnification (eq-m55-projection) | our own geometry |
| B | Segment the atrium and measure the segmentation error | m51 phantom |
| C | Register map points: landmarks, then surface | our own simulation |
| D | Predict target error from landmark error (eq-m55-tre) | published formula |
| E | Overlay the model on a fluoroscopy view | m49 phantom |
| F | The same registration on real anatomy (optional download) | Medical Segmentation Decathlon |
| G | An error and dose budget (eq-m55-budget) | parts B–E and published reference levels |

Registration itself is derived in m21; here we use it. The prediction of target registration error is from
Fitzpatrick, West and Maurer (1998), as restated by Danilchenko and Fitzpatrick (2011). Published magnitudes for
comparison come from Kistler et al. (2006) and Dong et al. (2006); reference levels for dose from Greffier et al.
(2026). The C-arm geometry, noise levels, landmark choices and dose rates are illustrative. Each part ends with
**checks**. Part F downloads a public data set (several hundred MB, CC BY-SA 4.0) once, as in m40 and m52; set
`EPML_SKIP_REAL_DATA=1` to skip it.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import capstone as C, ct, fluoro
from epml.data import DataUnavailable
from epml.ice import distance_to_ellipsoid

## Part A · C-arm projection and magnification

A point source sits a distance SOD before the isocenter and the detector a distance SID from the source. A point at
depth $y$ beyond the isocenter and lateral position $x$ lands at $u = x\,\mathrm{SID}/(\mathrm{SOD} + y)$
(eq-m55-projection). We use SOD = 750 mm and SID = 1100 mm, illustrative values.

In [ ]:
pts = np.array([[10.0, -60.0, 0.0], [10.0, 0.0, 0.0], [10.0, 60.0, 0.0]])
uv, mag = C.project(pts)
for p, u, m in zip(pts, uv[:, 0], mag):
    print(f"x = 10 mm at depth {p[1]:+.0f} mm: lands at {u:.2f} mm, magnification {m:.3f}")
ys = np.linspace(-100, 100, 201)
plt.plot(ys, C.project(np.c_[np.zeros_like(ys), ys, np.zeros_like(ys)] + [1, 0, 0])[1])
plt.xlabel("depth beyond the isocenter (mm)"); plt.ylabel("magnification"); plt.show()

In [ ]:
# Checks
assert abs(mag[1] - 1100 / 750) < 1e-12 and mag[0] > mag[1] > mag[2]

Structures nearer the source are magnified more; one projection cannot tell depth from size.

## Part B · Segment the atrium

The CT-like atrium from m51: a contrast-filled ellipsoid (semi-axes 25, 20 and 22 mm) with a wall in fat, at 1-mm
voxels. A threshold between wall and blood segments the blood pool; its boundary voxels are our surface. We measure
how far they lie from the true ellipsoid.

In [ ]:
semi = (25.0, 20.0, 22.0)
vol, ax, masks = ct.atrium_volume(voxel=1.0, semi_axes=semi, wall=(0.5, 3.5))
rng = np.random.default_rng(55)
noisy = vol + rng.normal(0, 30.0, vol.shape)          # illustrative image noise, HU
seg = noisy > 200.0                                    # between wall (~54 HU) and blood (400 HU)
from scipy.ndimage import binary_opening
seg = binary_opening(seg, iterations=1)
surf = C.surface_points(seg, ax)
d_seg = distance_to_ellipsoid(surf, semi)
seg_rms = float(np.sqrt(np.mean(d_seg ** 2)))
print(f"{len(surf)} surface points; distance to the true surface: mean {d_seg.mean():.2f} mm, RMS {seg_rms:.2f} mm")
z0 = np.argmin(abs(ax))
plt.imshow(noisy[:, :, z0].T, origin="lower", cmap="gray", vmin=-200, vmax=600, extent=[ax[0], ax[-1], ax[0], ax[-1]])
plt.contour(ax, ax, seg[:, :, z0].T, levels=[0.5], colors="C1")
plt.title("CT-like slice and segmentation"); plt.xlabel("x (mm)"); plt.ylabel("y (mm)"); plt.show()

In [ ]:
# Checks
assert seg_rms < 1.5

Boundary voxels sit about half a voxel inside the true surface; that bias enters every later step.

## Part C · Register map points: landmarks, then surface

The mapping system's frame differs from the image's by an unknown rigid transform. We simulate 300 map points on the
true atrial surface with 1.5 mm RMS localization error, four anatomical landmarks marked with 5 mm RMS error, and
four target points where lesions would be placed. All errors illustrative. Landmark registration (least squares,
m21) gives a start; surface registration (iterative closest point) refines it. We report what a system would show,
the fiducial error or mean surface distance, and what matters, the error at the targets.

In [ ]:
R_true = C.random_rotation(rng, 15)
t_true = np.array([8.0, -5.0, 6.0])
to_map = lambda p: C.apply(R_true, t_true, p)            # image frame -> map frame
map_pts = to_map(C.ellipsoid_points(semi, 300, rng)) + rng.normal(0, 1.5 / np.sqrt(3), (300, 3))
lm_img = np.array([[25, 0, 0], [-25, 0, 0], [0, 20, 0], [0, 0, 22]], float)
lm_map = to_map(lm_img) + rng.normal(0, 5.0 / np.sqrt(3), lm_img.shape)
targets_img = np.array([[0, -20, 0], [17.7, -14.1, 0], [-17.7, -14.1, 0], [0, 0, -22]], float)
targets_map = to_map(targets_img)

R1, t1 = C.kabsch(lm_map, lm_img)
fre_lm = C.rms(C.apply(R1, t1, lm_map) - lm_img)
tre_lm = np.linalg.norm(C.apply(R1, t1, targets_map) - targets_img, axis=1)
R2, t2, surf_dist = C.icp(map_pts, surf, R1, t1)
tre_surf = np.linalg.norm(C.apply(R2, t2, targets_map) - targets_img, axis=1)
print(f"landmarks: reported FRE {fre_lm:.1f} mm; error at targets {np.round(tre_lm, 1)} mm")
print(f"surface:   reported mean distance {surf_dist:.1f} mm; error at targets {np.round(tre_surf, 1)} mm")

In [ ]:
# Checks
assert surf_dist < tre_surf.mean()          # the reported surface distance understates the target error
assert tre_surf.mean() < tre_lm.mean()      # surface refinement helps here

The same pattern appears in the published numbers: a surface fit reported 2.3 mm against 6.4 mm for
landmarks (Kistler et al. 2006), and in animals the error at marked targets was 1.8–2.7 mm (Dong et al. 2006). Our
values come from illustrative noise levels and are not a reproduction of either study.

## Part D · Predict target error from landmark error

For isotropic localization error with RMS FLE on $N$ landmarks,
$\langle \mathrm{FRE}^2\rangle = (1 - 2/N)\langle\mathrm{FLE}^2\rangle$, whatever the layout, while
$\langle\mathrm{TRE}^2(r)\rangle \approx \frac{\langle\mathrm{FLE}^2\rangle}{N}\left(1 + \frac13\sum_k
\frac{d_k^2}{f_k^2}\right)$, where $d_k$ is the target's distance from principal axis $k$ of the landmarks and
$f_k^2$ the landmarks' mean squared distance from it (eq-m55-tre). We check it by Monte Carlo for three layouts at 2 mm RMS FLE (illustrative).

In [ ]:
targets = np.array([[0, 0, 0], [30, 0, 0], [-30, 10, 5], [0, 0, 30]], float)
rows = []
for kind in ("spread", "clustered", "nearly collinear"):
    F = C.fiducial_layout(kind, 6)
    pred = C.predicted_tre(F, targets, 2.0)
    sim, fre, tre0 = C.simulate_tre(F, targets, 2.0, rng, 3000)
    rows.append((kind, pred, sim, float(np.sqrt(np.mean(fre ** 2))), float(np.corrcoef(fre, tre0)[0, 1])))
    print(f"{kind:17s} predicted TRE {np.round(pred, 1)}  simulated {np.round(sim, 1)}  "
          f"FRE {rows[-1][3]:.2f} (predicted {C.predicted_fre(6, 2.0):.2f})  corr(FRE, TRE) {rows[-1][4]:+.2f}")
# with larger localization error the first-order prediction starts to fall short
F = C.fiducial_layout("clustered", 6)
ratio3 = C.simulate_tre(F, targets, 3.0, rng, 3000)[0] / C.predicted_tre(F, targets, 3.0)
print("clustered layout at 3 mm FLE, simulated / predicted:", np.round(ratio3, 2))

In [ ]:
# Checks
for kind, pred, sim, fre_rms, corr in rows:
    assert np.allclose(sim, pred, rtol=0.1)
    assert abs(fre_rms - C.predicted_fre(6, 2.0)) < 0.1
    assert abs(corr) < 0.1
assert ratio3.max() > 1.05                  # first order: accurate for small errors only

FRE is the same for every layout and tells you nothing about a particular case's TRE; the layout decides how
fast the error grows away from the landmarks. The formula is first order: when localization error is large compared
with the spread of the landmarks, the simulated error exceeds it.

## Part E · Overlay the model on a fluoroscopy view

We draw the registered atrial model on the simulated chest projection from m49. A registration error across the beam
shifts the overlay; an error along the beam barely shows in one view and appears when the C-arm is rotated.

In [ ]:
ph = fluoro.chest_phantom()
xs = np.linspace(-90, 110, 200); zs = np.linspace(-80, 70, 150)
X, Z = np.meshgrid(xs, zs)
img = -np.log(fluoro.projection(ph, X, Z, 1e5) / 1e5)
heart_center = np.array([15.0, -10.0, 10.0])
lines = C.ellipsoid_wireframe(semi, heart_center)
err = np.array([0.0, 6.0, 0.0])                    # 6 mm along the beam (front-back), illustrative
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for a, angle in zip(axs, (0.0, 60.0)):
    a.imshow(img, origin="lower", cmap="gray_r", extent=[xs[0], xs[-1], zs[0], zs[-1]], alpha=0.6 if angle else 1.0)
    for line in lines:
        uv0, m0 = C.project(line, angle_deg=angle)
        uv1, _ = C.project(line + err, angle_deg=angle)
        a.plot(uv0[:, 0] / m0.mean(), uv0[:, 1] / m0.mean(), color="0.4", lw=0.6)
        a.plot(uv1[:, 0] / m0.mean(), uv1[:, 1] / m0.mean(), color="C1", lw=0.8)
    seen, hidden = C.visible_error(err, angle)
    a.set_title(f"C-arm {angle:.0f} deg: {seen:.1f} mm visible, {hidden:.1f} mm hidden")
plt.tight_layout(); plt.show()
seen0, _ = C.visible_error(err, 0.0)
seen60, _ = C.visible_error(err, 60.0)
print(f"visible part of a 6-mm front-back error: {seen0:.1f} mm at 0 deg, {seen60:.1f} mm at 60 deg")

In [ ]:
# Checks
assert seen0 < 1e-9 and seen60 > 5.0

The background is the m49 phantom drawn at the isocenter scale only at 0°; at 60° it is shown faded for
orientation. One projection constrains two of three directions; a second view at an angle is what reveals the third.

## Part F · The same registration on real anatomy (optional download)

The Medical Segmentation Decathlon heart task (Antonelli et al. 2021; CC BY-SA 4.0) provides MRI volumes with the
left atrium labeled. We use one label's surface as the "image" and simulate map points on it with a known transform,
as in part C, to see the registration on a real atrial shape with its veins and appendage.

In [ ]:
try:
    import nibabel as nib
    from epml.data import fetch_msd_heart
    pairs = fetch_msd_heart()
    REAL = True
except DataUnavailable as err:
    print("Part F skipped:", err)
    REAL = False

if REAL:
    lp = pairs[0][1]
    li = nib.load(str(lp))
    spacing = np.array([float(z) for z in li.header.get_zooms()[:3]])
    lab = np.asarray(li.dataobj) > 0
    from epml.imaging import surface
    real_surf = np.argwhere(surface(lab)) * spacing
    real_surf = real_surf - real_surf.mean(0)
    rng_f = np.random.default_rng(7)
    pick = rng_f.choice(len(real_surf), size=min(400, len(real_surf)), replace=False)
    R_f, t_f = C.random_rotation(rng_f, 10), np.array([5.0, -4.0, 3.0])
    m_pts = C.apply(R_f, t_f, real_surf[pick]) + rng_f.normal(0, 1.5 / np.sqrt(3), (len(pick), 3))
    ext = [real_surf[np.argmax(real_surf[:, k])] for k in range(3)] + [real_surf[np.argmin(real_surf[:, 0])]]
    lm_i = np.array(ext)
    lm_m = C.apply(R_f, t_f, lm_i) + rng_f.normal(0, 5.0 / np.sqrt(3), lm_i.shape)
    tg_i = real_surf[rng_f.choice(len(real_surf), 20, replace=False)]
    Ra, ta = C.kabsch(lm_m, lm_i)
    Rb, tb, dist_f = C.icp(m_pts, real_surf, Ra, ta)
    tre_a = np.linalg.norm(C.apply(Ra, ta, C.apply(R_f, t_f, tg_i)) - tg_i, axis=1)
    tre_b = np.linalg.norm(C.apply(Rb, tb, C.apply(R_f, t_f, tg_i)) - tg_i, axis=1)
    print(f"{Path(lp).name}: voxel {np.round(spacing, 2)} mm, {len(real_surf)} surface points")
    print(f"landmarks: target error mean {tre_a.mean():.1f} mm; surface: reported {dist_f:.1f} mm, target error mean {tre_b.mean():.1f} mm")

In [ ]:
# Checks
if REAL:
    assert np.isfinite(tre_b).all() and dist_f >= 0

## Part G · An error and dose budget

Independent, zero-mean errors add in quadrature: $\sigma_{\text{total}} = \sqrt{\sum_i \sigma_i^2}$
(eq-m55-budget). We combine the segmentation error (part B), the registration error at targets (part C) and an
illustrative catheter localization error. For dose, the kerma-area product (KAP) adds the fluoroscopy rate times its
time and any acquisitions (m49); the rate and times here are illustrative, compared with French reference levels for
two procedures (Greffier et al. 2026).

In [ ]:
parts = {"segmentation (part B)": seg_rms, "registration at targets (part C)": C.rms(tre_surf[:, None]),
         "catheter localization (illustrative)": 1.0}
total = C.budget(parts.values())
for name, v in parts.items():
    print(f"{name:40s} {v:4.1f} mm  ({v ** 2 / total ** 2:.0%} of the variance)")
print(f"{'total (quadrature)':40s} {total:4.1f} mm;  simple sum would be {sum(parts.values()):.1f} mm")

kap = C.kap(rate_gycm2_per_min=0.6, minutes=8.0, acquisitions_gycm2=1.0)
reference = {"accessory pathway ablation": 4.10, "persistent AF ablation": 12.65}
print(f"KAP {kap:.1f} Gy cm2 (illustrative rate and times)")
for k, v in reference.items():
    print(f"  reference level, {k}: {v} Gy cm2 (Greffier et al. 2026)")

In [ ]:
# Checks
assert max(parts.values()) <= total <= sum(parts.values())

The largest term dominates a quadrature sum, so reducing it pays most; halving a small term barely moves the
total. The budget assumes the errors are independent and unbiased; a shared bias (for example a segmentation that is
always half a voxel inside) adds directly instead.

## Where to go next

- In part C, use landmarks only on one side of the atrium. What happens to the target error?
- In part D, how many spread-out landmarks give a 2-mm TRE at 30 mm from their center with 3-mm FLE?
- In part E, which C-arm angle shows a head-foot error best?
- In part G, which single improvement lowers the total most?